# Target acquisition

Take overview images, let the analysis find objects, then image the brightest few at high magnification.

It runs on the mock microscope. Run `python install.py` in this repository first.

## 1. Connect

In [ ]:
from zmart_controller import mic

mic.connect("mock", {"output_root": "images"})

## 2. Overview positions

Capture the position you want to image.

In [ ]:
overview_positions = []

In [ ]:
xyz = mic.get_xyz()["content"]
overview_positions.append([xyz["x"]["position"], xyz["y"]["position"], xyz["z"]["position"]])

In [ ]:
xyz = mic.get_xyz()["content"]
overview_positions.append([xyz["x"]["position"], xyz["y"]["position"], xyz["z"]["position"]])

## 3. States

Capture the overview settings and the target settings.

In [ ]:
mic.set_state({"changeable": {"objective": 1}})      # 10x: wide view
overview_state = {"changeable": mic.get_state()["content"]["changeable"]}

In [ ]:
mic.set_state({"changeable": {"objective": 3}})      # 40x: close-up
target_state = {"changeable": mic.get_state()["content"]["changeable"]}

## 4. Analysis

Set up the analysis that finds the objects.

In [ ]:
from zmart_analysis import Engine

ZMART_ANALYSIS = "../../work/ZMART-analysis"

engine = Engine()
engine.register(
    "find_objects",
    f"{ZMART_ANALYSIS}/workflows/object_analysis/pipelines/object_analysis_watershed.yaml",
    "analysis_output",
)

## 5. The loop

Image the overview positions, find the objects, then image the brightest ones.

The image size (64 x 64 pixels) and orientation `[[1, 0], [0, 1]]` are the mock's.

In [ ]:
import time

N_TARGETS = 5

# 1. Overview: take an image at each position and send it to the analysis.
mic.set_state(overview_state)
jobs = {}
for i, (x, y, z) in enumerate(overview_positions):
    mic.set_xyz(x, y, z)
    image = mic.acquire(position_label=f"overview{i}")["content"]
    pixel_um = mic.get_state()["content"]["observed"]["pixel_size"]["x"]
    jobs[i] = engine.submit("find_objects", {
        "image_path": image["files"][0],
        "tile_id": [f"overview{i}"],
        "tile_stage_xy_um": [x, y],
        "source_pixel_size_um": [pixel_um, pixel_um],
        "source_image_size_px": [64, 64],
        "image_to_stage": [[1, 0], [0, 1]],
    })

# 2. Wait for the analysis, then choose the brightest objects of each overview.
targets = []
for i, job in jobs.items():
    while (entry := engine.status("find_objects")[job])["state"] not in ("done", "failed"):
        time.sleep(0.2)
    assert entry["state"] == "done", entry.get("error")
    table = entry["output"]["object_analysis"]["objects"]["properties"]
    brightest = sorted(range(len(table["label"])), key=lambda k: -table["intensity_mean"][k])
    for k in brightest[:N_TARGETS]:
        targets.append((f"overview{i}_object{table['label'][k]}",
                        table["stage_x_um"][k], table["stage_y_um"][k], overview_positions[i][2]))

# 3. Targets: switch to the close-up state and visit each chosen object.
mic.set_state(target_state)
for label, x, y, z in targets:
    mic.set_xyz(x, y, z)
    mic.acquire(position_label=label)

Close the connections.

In [ ]:
engine.shutdown()
mic.disconnect()